# 08 · Incertidumbre: intervalos de confianza por bootstrap

Los ahorros de `04` y `06` son estimaciones puntuales sobre 2,467 productos. Aquí se remuestrean **productos con reemplazo** (2,000 réplicas) y en cada réplica se repite todo el cálculo: nivel de servicio óptimo de cada modelo, costo total e inventario interpolado a fill rate de 95%.

Usa los resultados por producto que guarda `06_ablacion_censura` (`data/processed/ablacion_por_producto_CA_1.parquet`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PROC = ROOT / "data" / "processed"
REP = ROOT / "reports"
H_ANUAL, MARGEN, N_DIAS = 0.25, 0.30, 175   # 189 días simulados − 14 de arranque
B, FR_OBJ = 2000, 0.95

pp = pd.read_parquet(PROC / "ablacion_por_producto_CA_1.parquet")
modelos = ["LGB", "AutoETS_imp", "TSB_imp", "AutoETS", "TSB", "SeasonalNaive"]
alphas = sorted(pp["alpha"].unique())
ids = sorted(pp["id"].unique())

def cubo(col):
    """Arreglo (modelo, alpha, producto) de una métrica."""
    return (pp.set_index(["modelo", "alpha", "id"])[col]
              .unstack("id").reindex(columns=ids)
              .reindex(pd.MultiIndex.from_product([modelos, alphas])).to_numpy()
              .reshape(len(modelos), len(alphas), len(ids)))

X = {c: cubo(c) for c in ["inv_usd_dia", "perdido_usd", "vendido_u", "demanda_u"]}

# Control: la muestra completa reproduce las tablas agregadas de 06
tabla06 = pd.read_csv(REP / "ablacion_censura_CA_1.csv").set_index(["modelo", "alpha"])
costo = X["inv_usd_dia"].sum(-1) * H_ANUAL / 365 + X["perdido_usd"].sum(-1) * MARGEN
assert np.allclose(costo, tabla06["costo_total"].reindex(pd.MultiIndex.from_product([modelos, alphas]))
                   .to_numpy().reshape(costo.shape), rtol=1e-9)
print(f"{len(ids):,} productos × {len(modelos)} modelos × {len(alphas)} niveles de servicio")

2,467 productos × 6 modelos × 7 niveles de servicio


In [2]:
def metricas(W):
    """W: (réplicas, productos) con el número de veces que entra cada producto."""
    agg = {c: np.einsum("bp,map->bma", W, x) for c, x in X.items()}       # (B, modelo, alpha)
    costo = agg["inv_usd_dia"] * H_ANUAL / 365 + agg["perdido_usd"] * MARGEN
    fill = agg["vendido_u"] / agg["demanda_u"]
    inv = agg["inv_usd_dia"] / N_DIAS
    costo_opt = costo.min(axis=2)                                           # (B, modelo)
    inv95 = np.full(costo_opt.shape, np.nan)
    for b in range(W.shape[0]):
        for m in range(len(modelos)):
            o = np.argsort(fill[b, m])
            if fill[b, m, o[0]] <= FR_OBJ <= fill[b, m, o[-1]]:
                inv95[b, m] = np.interp(FR_OBJ, fill[b, m, o], inv[b, m, o])
    return costo_opt, inv95

comparaciones = {
    "Censura · TSB (demanda corregida vs. ventas)": ("TSB_imp", "TSB"),
    "Censura · AutoETS (demanda corregida vs. ventas)": ("AutoETS_imp", "AutoETS"),
    "Modelo · LightGBM vs. AutoETS corregido": ("LGB", "AutoETS_imp"),
    "Modelo · LightGBM vs. TSB corregido": ("LGB", "TSB_imp"),
    "Original · LightGBM vs. TSB con ventas": ("LGB", "TSB"),
}

c_full, i_full = metricas(np.ones((1, len(ids))))
rng = np.random.default_rng(7)
W = rng.multinomial(len(ids), np.full(len(ids), 1 / len(ids)), size=B).astype(float)
c_bs, i_bs = metricas(W)

idx = {m: i for i, m in enumerate(modelos)}
filas = []
for nombre, (a, b) in comparaciones.items():
    for metrica, full, bs in [("Costo total óptimo", c_full, c_bs), ("Inventario a fill rate 95%", i_full, i_bs)]:
        cambio = (bs[:, idx[a]] / bs[:, idx[b]] - 1) * 100
        filas.append({"comparacion": nombre, "metrica": metrica,
                      "cambio_pct": (full[0, idx[a]] / full[0, idx[b]] - 1) * 100,
                      "ic95_inf": np.nanpercentile(cambio, 2.5), "ic95_sup": np.nanpercentile(cambio, 97.5),
                      "prob_ahorro": np.nanmean(cambio < 0)})
ic = pd.DataFrame(filas)
ic.to_csv(REP / "bootstrap_ic_CA_1.csv", index=False)
ic.round({"cambio_pct": 1, "ic95_inf": 1, "ic95_sup": 1, "prob_ahorro": 3})   # negativo = el primero cuesta / guarda menos

,comparacion,metrica,cambio_pct,ic95_inf,ic95_sup,prob_ahorro
0,Censura · TSB (demanda corregida vs. ventas),Costo total óptimo,-27.6,-31.8,-23.5,1.000
1,Censura · TSB (demanda corregida vs. ventas),Inventario a fill rate 95%,-23.8,-34.5,-18.8,1.000
2,Censura · AutoETS (demanda corregida vs. ventas),Costo total óptimo,-24.7,-29.7,-19.7,1.000
3,Censura · AutoETS (demanda corregida vs. ventas),Inventario a fill rate 95%,-23.8,-33.4,-15.5,1.000
4,Modelo · LightGBM vs. AutoETS corregido,Costo total óptimo,2.6,-1.4,6.0,0.088
5,Modelo · LightGBM vs. AutoETS corregido,Inventario a fill rate 95%,-0.3,-4.9,3.9,0.550
6,Modelo · LightGBM vs. TSB corregido,Costo total óptimo,2.1,-1.0,5.3,0.091
7,Modelo · LightGBM vs. TSB corregido,Inventario a fill rate 95%,-1.8,-5.8,2.0,0.829
8,Original · LightGBM vs. TSB con ventas,Costo total óptimo,-26.1,-31.4,-21.1,1.000
9,Original · LightGBM vs. TSB con ventas,Inventario a fill rate 95%,-25.1,-35.4,-19.6,1.000


## Conclusiones

| Comparación | Costo total óptimo | Inventario a fill rate 95% |
|---|---|---|
| **Censura** · TSB con demanda corregida vs. con ventas | **−27.6%** (IC 95%: −31.8% a −23.5%) | **−23.8%** (−34.5% a −18.8%) |
| **Censura** · AutoETS con demanda corregida vs. con ventas | **−24.7%** (−29.7% a −19.7%) | **−23.8%** (−33.4% a −15.5%) |
| **Modelo** · LightGBM vs. AutoETS corregido | +2.6% (−1.4% a +6.0%) | −0.3% (−4.9% a +3.9%) |
| **Modelo** · LightGBM vs. TSB corregido | +2.1% (−1.0% a +5.3%) | −1.8% (−5.8% a +2.0%) |

- El efecto de corregir la censura es grande y robusto: en el 100% de las réplicas reduce costo e inventario, con ambos modelos clásicos.
- La diferencia entre LightGBM y un método clásico alimentado con demanda corregida **no es significativa**: los intervalos incluyen el cero.
- **Limitación:** el bootstrap remuestrea productos; no captura la incertidumbre temporal (otro semestre podría dar otro resultado) ni la de la calibración del stock de seguridad, que se mantiene fija.